#**BRAZILIAN PORTUGESE**

In [9]:
import pandas as pd

# Raw URL from GitHub
URL = "https://raw.githubusercontent.com/Selena8877/ML-Group-45-Project/refs/heads/main/dataset_ideacao%20(3).csv"

df = pd.read_csv(URL, encoding="utf-8")

print("Shape (rows, columns):", df.shape)
print("Columns:", df.columns.tolist())
pd.set_option("display.max_colwidth", 200)
df.head(10)

Shape (rows, columns): (3788, 2)
Columns: ['text', 'target']


,text,target
0,Aquela vontade de acabar com a minha vida voltou,1
1,to triste e com vontade de acabar com a minha vida,1
2,Corinthians ta querendo acabar com minha vida mesmo,0
3,Alguém poderia por favor me dar um tiro a acabar com a minha vida,1
4,TAYLOR TU VAI acabar com a minha vida MULHER,0
5,Eu vou assistir o jogo todo puto mais uma vez. Esse velho maldito vai conseguir acabar com a minha vida,0
6,Alguem pode me.indicar uma série uma boa q vai acabar com a minha vida social pelas próximas semanas??,0
7,tenho 76 slides p estudar obg faculdade por acabar com a minha vida social,0
8,eu nao concordei com um tweet da fernanda agora ela desenterrou um hit meu pra acabar com a minha vida acho lindo a amizade pós-moderna,0
9,sessão de foto pra acabar com a minha vida esse fudido https://twitter.com/h1ghrarchive/status/1117462044226101253 …,0


In [10]:

# clean text, remove empty/duplicate tweets

import re, html

log = []
def log_step(name, data):
    log.append({"step": name, "rows": len(data),
                "label_1": int((data["label"] == 1).sum()),   # literal suicidal ideation
                "label_0": int((data["label"] == 0).sum())})  # not

df = df.rename(columns={"target": "label", "text": "tweet"})   # same names as the other languages
log_step("0. raw", df)

# 1. remove empty tweets
df = df.dropna(subset=["tweet"])
df["tweet"] = df["tweet"].astype(str)
df = df[df["tweet"].str.strip() != ""]
log_step("1. drop empty", df)

# 2. clean text (same rules as Thai/Chinese)
def clean_text(t):
    t = html.unescape(t)                                               # &amp; -> &
    t = re.sub(r"[\u200b\u200e\u200f\u202a-\u202e\u2066-\u2069]", "", t)  # invisible characters
    t = re.sub(r"^RT\s+", "", t)                                       # retweet marker
    t = re.sub(r"https?://\S+|www\.\S+", "", t)                        # links
    t = re.sub(r"@\w+", "[USER]", t)                                   # usernames -> [USER]
    t = re.sub(r"#\S+", "", t)                                         # hashtags (same as Thai/Chinese)
    t = re.sub(r"\s+", " ", t).strip()                                 # extra spaces
    return t

df["tweet"] = df["tweet"].apply(clean_text)
df = df[df["tweet"] != ""]
log_step("2. clean text", df)

# 3. remove contradictory tweets (same text, different labels)
label_counts = df.groupby("tweet")["label"].nunique()
df = df[~df["tweet"].isin(label_counts[label_counts > 1].index)]
log_step("3. drop conflicting", df)

# 4. remove duplicates
df = df.drop_duplicates(subset="tweet")
log_step("4. drop duplicates", df)

print(pd.DataFrame(log))

# lengths
df["n_words"] = df["tweet"].str.split().str.len()
print("\nWords per tweet, by label:")
print(df.groupby("label")["n_words"].describe())

print("\nHow many tweets per label would survive a minimum length:")
for m in [10, 15, 20]:
    kept = df[df["n_words"] >= m]["label"].value_counts()
    print(f"  >= {m} words ->  label 1: {kept.get(1,0)},  label 0: {kept.get(0,0)}")

# does the phrase appear in both classes?
df["has_phrase"] = df["tweet"].str.lower().str.contains("acabar com a minha vida|acabar com minha vida")
print("\nContains 'acabar com (a) minha vida', by label:")
print(pd.crosstab(df["label"], df["has_phrase"]))

                  step  rows  label_1  label_0
0               0. raw  3788     1097     2691
1        1. drop empty  3788     1097     2691
2        2. clean text  3788     1097     2691
3  3. drop conflicting  3772     1089     2683
4   4. drop duplicates  2940      802     2138

Words per tweet, by label:
        count       mean        std  min  25%   50%   75%   max
label                                                          
0      2138.0  17.785313  11.847227  2.0  9.0  14.0  22.0  70.0
1       802.0  11.820449  10.025445  1.0  6.0   8.0  14.0  58.0

How many tweets per label would survive a minimum length:
  >= 10 words ->  label 1: 340,  label 0: 1559
  >= 15 words ->  label 1: 191,  label 0: 1050
  >= 20 words ->  label 1: 113,  label 0: 670

Contains 'acabar com (a) minha vida', by label:
has_phrase  False  True 
label                   
0            2092     46
1             789     13


In [11]:
#keep >= 15 words, balanced sample

MIN_WORDS, N_PER_CLASS, SEED = 15, 50, 42    # same settings as the other languages

# Save all cleaned tweets BEFORE the length filter (for charts)
df.to_csv("portuguese_clean_before_length_filter.csv", index=False, encoding="utf-8-sig")

df_long = df[df["n_words"] >= MIN_WORDS].copy()
log_step(f"5. keep >= {MIN_WORDS} words", df_long)

sample = pd.concat([
    df_long[df_long["label"] == 1].sample(n=N_PER_CLASS, random_state=SEED),
    df_long[df_long["label"] == 0].sample(n=N_PER_CLASS, random_state=SEED),
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

sample.insert(0, "id", [f"pt_{i+1:03d}" for i in range(len(sample))])
sample.insert(1, "language", "Portuguese")
log_step("6. balanced sample", sample)

# split into prefix/middle/suffix
# (same as Arabic: Portuguese has spaces, so .split() works)
def split_into_thirds(text):
    words = text.split()
    n = len(words)
    base, extra = n // 3, n % 3
    sizes = [base + (1 if i < extra else 0) for i in range(3)]
    prefix = words[:sizes[0]]
    middle = words[sizes[0]:sizes[0] + sizes[1]]
    suffix = words[sizes[0] + sizes[1]:]
    return " ".join(prefix), " ".join(middle), " ".join(suffix)

sample[["prefix", "middle", "suffix"]] = sample["tweet"].apply(
    lambda t: pd.Series(split_into_thirds(t)))
sample = sample.rename(columns={"tweet": "full"})

rebuilt = sample["prefix"] + " " + sample["middle"] + " " + sample["suffix"]
print("All splits correct:", (rebuilt == sample["full"]).all())

# Is there still a length gap in the sample?
print("\nWords per tweet in the SAMPLE, by label:")
print(sample.groupby("label")["n_words"].median())

cols = ["id", "language", "label", "full", "prefix", "middle", "suffix", "n_words", "has_phrase"]
sample = sample[cols]

# save
sample.to_csv("portuguese_sample_split.csv", index=False, encoding="utf-8-sig")
df_long.to_csv("portuguese_clean_full.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(log).to_csv("portuguese_cleaning_log.csv", index=False)

print("\n", pd.DataFrame(log))
sample[["id", "label", "full", "prefix", "middle", "suffix"]].head(5)

All splits correct: True

Words per tweet in the SAMPLE, by label:
label
0    20.5
1    20.5
Name: n_words, dtype: float64

                   step  rows  label_1  label_0
0               0. raw  3788     1097     2691
1        1. drop empty  3788     1097     2691
2        2. clean text  3788     1097     2691
3  3. drop conflicting  3772     1089     2683
4   4. drop duplicates  2940      802     2138
5  5. keep >= 15 words  1241      191     1050
6   6. balanced sample   100       50       50


,id,label,full,prefix,middle,suffix
0,pt_001,0,"Quem chorou duas horas sem parar? Sim eu, pq minha família vai me matar com cada sintoma diferente que parece :x",Quem chorou duas horas sem parar? Sim,"eu, pq minha família vai me matar",com cada sintoma diferente que parece :x
1,pt_002,0,"Eu to há mais de um mês sem ir na barbearia, pelo amor...isso é suicídio social",Eu to há mais de um,"mês sem ir na barbearia,",pelo amor...isso é suicídio social
2,pt_003,0,Amg e a missao suicida dele na batalha dos bastardos que ele levou todo mundo pra morrer ali foi um odio viu ramsen pq n matou logo aquele merda,Amg e a missao suicida dele na batalha dos bastardos,que ele levou todo mundo pra morrer ali foi um,odio viu ramsen pq n matou logo aquele merda
3,pt_004,1,Queria poder ter uma forma simples de me despedir dos meus amigos de uma maneira que eles não percebam oque pretendo fazer. A minha carta de suicídio tá pronta e nela eu peço perdão a poucas pesso...,Queria poder ter uma forma simples de me despedir dos meus amigos de uma maneira que eles,não percebam oque pretendo fazer. A minha carta de suicídio tá pronta e nela eu peço perdão,a poucas pessoas. outras eu agradeço e outras eu agradeço por me fazerem chegar a isto
4,pt_005,1,[USER] serio eu vou me matar. eu n posso nem comer por causa dessa bct,[USER] serio eu vou me,matar. eu n posso nem,comer por causa dessa bct
